# Create Spark session

In [1]:
from pyspark.sql import SparkSession


spark = SparkSession.builder \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "4g") \
    .appName("Open food facts") \
    .getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/05/26 21:26:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


25/05/26 21:26:42 WARN GarbageCollectionMetrics: To enable non-built-in garbage collector(s) List(G1 Concurrent GC), users should configure it(them) to spark.eventLog.gcMetrics.youngGenerationGarbageCollectors or spark.eventLog.gcMetrics.oldGenerationGarbageCollectors


# Read dataset

In [2]:
df = spark.read.csv("/Users/mac/Desktop/Spark-Recommendation-System/data/en.openfoodfacts.org.products.csv.gz", sep="\t" ,header=True)

## Schema

In [3]:
df.printSchema()

root
 |-- code: string (nullable = true)
 |-- url: string (nullable = true)
 |-- creator: string (nullable = true)
 |-- created_t: string (nullable = true)
 |-- created_datetime: string (nullable = true)
 |-- last_modified_t: string (nullable = true)
 |-- last_modified_datetime: string (nullable = true)
 |-- last_modified_by: string (nullable = true)
 |-- last_updated_t: string (nullable = true)
 |-- last_updated_datetime: string (nullable = true)
 |-- product_name: string (nullable = true)
 |-- abbreviated_product_name: string (nullable = true)
 |-- generic_name: string (nullable = true)
 |-- quantity: string (nullable = true)
 |-- packaging: string (nullable = true)
 |-- packaging_tags: string (nullable = true)
 |-- packaging_en: string (nullable = true)
 |-- packaging_text: string (nullable = true)
 |-- brands: string (nullable = true)
 |-- brands_tags: string (nullable = true)
 |-- brands_en: string (nullable = true)
 |-- categories: string (nullable = true)
 |-- categories_tags: s

## Exemples

In [4]:
df.show(5)

25/05/26 21:27:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------+--------------------+-------+----------+--------------------+---------------+----------------------+----------------+--------------+---------------------+--------------------+------------------------+------------+------------------+---------+--------------+------------+--------------+--------------+-----------------+--------------+--------------------+--------------------+--------------------+-------+------------+----------+--------------------+-------------------------+--------------------+--------------------+--------------------+---------+--------------+------------------------+------+-----------+---------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------------+---------+------------+------+-----------+---------+------------+----------------+-----------------+-----------+---------+--------------------+--------------------+----------------+----------------+----------+-------------+--

## number of columns

In [5]:
len(df.columns)

209

## Filter based on the countries

In [6]:
df.groupBy("countries_tags").count().orderBy("count", ascending=False).show(truncate=False)

+-------------------------+-------+
|countries_tags           |count  |
+-------------------------+-------+
|en:france                |1038588|
|en:united-states         |707614 |
|en:spain                 |312438 |
|en:germany               |267160 |
|en:italy                 |232091 |
|en:united-kingdom        |135501 |
|en:canada                |94182  |
|en:switzerland           |73783  |
|en:belgium               |68917  |
|en:ireland               |63826  |
|en:australia             |52118  |
|en:world                 |34571  |
|en:netherlands           |32254  |
|en:brazil                |23829  |
|NULL                     |23408  |
|en:japan                 |22859  |
|en:united-states,en:world|22694  |
|en:russia                |22487  |
|en:poland                |20803  |
|en:norway                |19723  |
+-------------------------+-------+
only showing top 20 rows



# divide our dataset

- we will choose just a subset of our full dataset to use, this is due to the lack of ressources

In [8]:
from pyspark.sql.functions import  col

filtered_df = df.filter(col("countries_tags").like("%en:france%"))


In [9]:
filtered_df.show(5)

+------------+--------------------+-------+----------+--------------------+---------------+----------------------+----------------+--------------+---------------------+--------------------+------------------------+------------+------------------+---------+--------------+------------+--------------+--------------+-----------------+--------------+--------------------+--------------------+-------------------+-------+------------+----------+--------------------+-------------------------+--------------------+--------------------+--------------------+---------+--------------+------------------------+------+-----------+---------------+------+--------------------+--------------------+--------------------+--------------------+--------------------+-------------------------+---------+------------+------+-----------+---------+------------+----------------+-----------------+-----------+---------+--------------------+--------------------+----------------+----------------+----------+-------------+---

In [10]:
filtered_df.count()

1148555

### Save new df

In [ ]:
filtered_df.write.option("header", True).mode("overwrite").csv("/Users/mac/Desktop/Spark-Recommendation-System/data/france_df.csv")